# 00 · Окружение и пути
Python 3.11/3.12. Запускать из клона Avito_test на сервере с GPU.
Ноутбуки 01 → 04 → 05 дают полный маршрут подготовки, обучения и submission.
Используем одно основное Python-окружение.

Установка ниже выключена по умолчанию: сначала проверь существующее окружение сервера.
Для отдельного окружения: `python -m venv .venv`, затем установи зависимости его Python.
Выбери это окружение как ядро Jupyter. Интернет используется только для скачивания
открытых данных, библиотек и весов; предсказания выполняются локально.


In [ ]:
import sys, subprocess, platform
from pathlib import Path


def find_project_root(start=None):
    from pathlib import Path
    import os

    current = Path(start or Path.cwd()).resolve()
    explicit = os.environ.get("AVITO_PROJECT_ROOT")
    candidates = (
        [Path(explicit).expanduser().resolve()] if explicit else [current, *current.parents]
    )
    for candidate in candidates:
        marker = (candidate / "avito_project.json").is_file()
        layout = (candidate / "notebooks" / "_shared.ipynb").is_file() and (
            candidate / "requirements.txt"
        ).is_file()
        if marker or layout:
            return candidate
    raise RuntimeError(
        f"Не найден корень Avito_test. Текущая папка: {current}. "
        "Перенеси notebooks/ целиком, requirements.txt и avito_project.json в папку проекта. "
        "На VM выполни в отдельной ячейке: %cd /home/slomauh/ml/Avito_test . "
        "Для другого расположения задай переменную окружения AVITO_PROJECT_ROOT. "
        + (f"Сейчас AVITO_PROJECT_ROOT={explicit!r}; проверь этот путь." if explicit else "")
    )


ROOT = find_project_root()
print("Python:", sys.executable, sys.version)
print("Система:", platform.platform())
print("Корень:", ROOT)
try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False).stdout)
except FileNotFoundError:
    print("nvidia-smi отсутствует: GPU-драйвер ещё не установлен или это CPU-машина.")


In [ ]:
# GPU-PyTorch уже настроен на VM: его не переустанавливаем.
INSTALL = False  # True устанавливает только остальные зависимости проекта
if INSTALL:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")]
    )
    print("Зависимости установлены. Перезапусти ядро перед следующей ячейкой.")


На VM выбери существующее ядро `/home/slomauh/ml/venv/bin/python`.
PyTorch с CUDA 12.1 уже видит T4 по проверке пользователя; оставляем его установленную версию.
`requirements.txt` не содержит torch/torchvision и не переустанавливает их напрямую.
Версии фактического окружения сохраняются с каждым экспериментом.
Для запуска этих ноутбуков подключение к ChatGPT не требуется.


In [ ]:
from pathlib import Path


def find_project_root(start=None):
    from pathlib import Path
    import os

    current = Path(start or Path.cwd()).resolve()
    explicit = os.environ.get("AVITO_PROJECT_ROOT")
    candidates = (
        [Path(explicit).expanduser().resolve()] if explicit else [current, *current.parents]
    )
    for candidate in candidates:
        marker = (candidate / "avito_project.json").is_file()
        layout = (candidate / "notebooks" / "_shared.ipynb").is_file() and (
            candidate / "requirements.txt"
        ).is_file()
        if marker or layout:
            return candidate
    raise RuntimeError(
        f"Не найден корень Avito_test. Текущая папка: {current}. "
        "Перенеси notebooks/ целиком, requirements.txt и avito_project.json в папку проекта. "
        "На VM выполни в отдельной ячейке: %cd /home/slomauh/ml/Avito_test . "
        "Для другого расположения задай переменную окружения AVITO_PROJECT_ROOT. "
        + (f"Сейчас AVITO_PROJECT_ROOT={explicit!r}; проверь этот путь." if explicit else "")
    )


ROOT = find_project_root()
get_ipython().run_line_magic("run", str(ROOT / "notebooks" / "_shared.ipynb"))
print("Проект:", ROOT)


In [ ]:
for folder in ["data/gt", "data/processed", "data/test", "runs", "submissions"]:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
info = environment()
print(json.dumps(info, ensure_ascii=False, indent=2))
if torch.cuda.is_available():
    x = torch.randn(128, 128, device="cuda")
    print("GPU-проверка:", (x @ x).shape)
else:
    print("CPU подходит для проверки кода; перед полным обучением подключи GPU-ядро.")
save_json(ROOT / "runs" / "environment_setup.json", info)


Ожидаемые файлы данных:
- `data/train/` и `data/validation/`: исходные изображения HierText;
- `data/gt/train.jsonl.gz` и `data/gt/validation.jsonl.gz`: соответствующая разметка
  (распакованные `.jsonl` также поддерживаются);
- `data/test/`: только тестовые изображения Авито;
- `data/sample_submission.csv`: шаблон организаторов.

Папка `avito` переименована в `data`; дополнительная вложенность `raw/hiertext` не нужна.
Для уже загруженных изображений оставь `DOWNLOAD=False`, `EXTRACT=False` в 01.
Одних JPG недостаточно: для правильной ориентации кропов требуется разметка HierText.
Тест Авито не используем для обучения или настройки параметров.
